# Analysis of ZeroSumNormal Constraint on Fourier Seasonality

This notebook analyzes the `ZeroSumNormal` constraint used for Fourier coefficients in CausalPy's `StateSpaceTimeSeries` model.

**Key Findings:**
1. **Mathematical limitation**: Certain signals (proportional to $g(t)$) cannot be represented by zero-sum constrained coefficients
2. **Practical mitigation**: The Kalman filter's dynamic state evolution compensates for this limitation with sufficient data
3. **When it matters**: Short pre-treatment periods (<12 months) can lead to model instability

**Tested with:** CausalPy 0.7.0, micromamba environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Check CausalPy availability
try:
    import causalpy as cp
    CAUSALPY_AVAILABLE = True
    print(f"CausalPy version: {cp.__version__}")
except ImportError:
    CAUSALPY_AVAILABLE = False
    print("CausalPy not available - using simulation mode")

## Part 1: Mathematical Foundation

### The Problem in CausalPy

In `causalpy/pymc_models.py`, the `StateSpaceTimeSeries` model uses:

```python
_annual_seasonal = pm.ZeroSumNormal("params_freq", sigma=80, dims=annual_dims)
```

This constrains the Fourier coefficients $\theta = (a_1, b_1, a_2, b_2, \ldots, a_6)$ to satisfy $\sum_i \theta_i = 0$.

### The Unrepresentable Signal $g(t)$

This constraint makes it impossible to represent signals proportional to:

$$g(t) = \begin{cases} 
n = S/2 & \text{if } t = 0 \\
0 & \text{if } t \text{ even}, t \neq 0 \\
\cot\left(\frac{\pi t}{S}\right) - 1 & \text{if } t \text{ odd}
\end{cases}$$

In [ ]:
def g_unrepresentable(t, S=12):
    """
    Closed-form for the unrepresentable signal.
    
    This is the function orthogonal to all zero-sum constrained Fourier representations.
    It corresponds to Fourier coefficients θ = (1, 1, 1, ..., 1).
    """
    t = np.asarray(t) % S
    result = np.zeros_like(t, dtype=float)
    result[t == 0] = S // 2
    odd_mask = (t % 2 == 1)
    result[odd_mask] = 1.0 / np.tan(np.pi * t[odd_mask] / S) - 1.0
    return result

# Display g(t) for S=12 (monthly data with annual seasonality)
S = 12
t_month = np.arange(S)
g = g_unrepresentable(t_month, S)

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

print("The unrepresentable signal g(t) for S=12:")
print("=" * 50)
for t, name, val in zip(t_month, month_names, g):
    formula = f"cot(π·{t}/12)-1" if t % 2 == 1 else ("6" if t == 0 else "0")
    print(f"  g({name:3s}) = {val:7.3f}  [{formula}]")
print("=" * 50)
print(f"  Mean: {np.mean(g):.6f} (zero mean - this IS a valid seasonal pattern!)")
print(f"  Var:  {np.var(g):.2f}")

In [ ]:
# Visualize g(t)
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['crimson' if abs(gi) > 0.01 else 'lightgray' for gi in g]
ax.bar(t_month, g, color=colors, edgecolor='black', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('g(t)', fontsize=12)
ax.set_title('The Unrepresentable Signal g(t) — Cannot Be Fitted by ZeroSumNormal Model', fontsize=14)
ax.set_xticks(t_month)
ax.set_xticklabels(month_names)

# Annotate extreme values
ax.annotate(f'g(Jan) = {g[0]:.0f}', xy=(0, g[0]), xytext=(1.5, g[0]+0.5),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=11)
ax.annotate(f'g(Dec) = {g[11]:.1f}', xy=(11, g[11]), xytext=(9, g[11]-1),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=11)

plt.tight_layout()
plt.show()

## Part 2: The Residual Structure

### Key Theorem

For **any** seasonal signal $y(t) = X\theta$ with Fourier coefficients $\theta$, the residual from zero-sum constrained fitting is:

$$\boxed{\text{residual} = \overline{\theta} \cdot g(t)}$$

where $\overline{\theta} = \frac{1}{11}\sum_i \theta_i$ is the mean of the coefficients.

In [ ]:
def build_fourier_basis(t, S=12):
    """Build Fourier basis matching CausalPy's FrequencySeasonality."""
    n = S // 2
    basis = []
    for j in range(1, n):  # j = 1 to 5
        basis.append(np.cos(2 * np.pi * j * t / S))
        basis.append(np.sin(2 * np.pi * j * t / S))
    basis.append(np.cos(2 * np.pi * n * t / S))  # Nyquist
    return np.column_stack(basis)

def fit_with_zero_sum_constraint(y, X):
    """Simulate fitting with ZeroSumNormal constraint."""
    # Unconstrained OLS
    theta_ols, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    # Project to zero-sum (what ZeroSumNormal enforces)
    theta_zs = theta_ols - np.mean(theta_ols)
    y_fit = X @ theta_zs
    return theta_zs, y_fit, theta_ols

# Example: simple sinusoidal pattern
X = build_fourier_basis(t_month, S)
y_simple = 2 * np.cos(2 * np.pi * t_month / S) + 0.5 * np.sin(2 * np.pi * 2 * t_month / S)

theta_zs, y_fit, theta_ols = fit_with_zero_sum_constraint(y_simple, X)
residual = y_simple - y_fit

# Verify: residual = mean(θ) * g(t)
mean_theta = np.mean(theta_ols)
predicted_residual = mean_theta * g

print("Example: y = 2cos(ωt) + 0.5sin(2ωt)")
print(f"  True coefficients sum: {np.sum(theta_ols):.2f}")
print(f"  True coefficients mean: {mean_theta:.4f}")
print(f"\nResidual verification:")
print(f"  residual = mean(θ) · g(t) = {mean_theta:.4f} · g(t)")
print(f"  Match: {np.allclose(residual, predicted_residual)}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(t_month, y_simple, 'ko-', markersize=8, linewidth=2)
ax.set_title('Target Signal', fontsize=12)
ax.set_xlabel('Month')
ax.set_xticks(t_month)
ax.set_xticklabels(month_names, rotation=45)

ax = axes[1]
ax.plot(t_month, y_simple, 'ko-', markersize=8, linewidth=2, label='Target')
ax.plot(t_month, y_fit, 'b^--', markersize=6, linewidth=2, label='Zero-sum fit')
ax.set_title(f'Fit (RMSE = {np.sqrt(np.mean(residual**2)):.3f})', fontsize=12)
ax.legend()
ax.set_xticks(t_month)
ax.set_xticklabels(month_names, rotation=45)

ax = axes[2]
ax.bar(t_month, residual, color='crimson', alpha=0.7, edgecolor='black', label='Actual residual')
ax.plot(t_month, predicted_residual, 'k--', linewidth=2, label=f'{mean_theta:.3f}·g(t)')
ax.set_title(f'Residual = mean(θ)·g(t)', fontsize=12)
ax.legend()
ax.set_xticks(t_month)
ax.set_xticklabels(month_names, rotation=45)

plt.tight_layout()
plt.show()

## Part 3: Impact on CausalPy Treatment Effect Estimation

In CausalPy's `InterruptedTimeSeries`, the `StateSpaceTimeSeries` model:
1. Fits the pre-intervention data to learn the seasonal pattern
2. Extrapolates a counterfactual to the post-intervention period
3. Estimates treatment effect as observed minus counterfactual

If the true seasonal pattern has $\overline{\theta} \neq 0$, the **counterfactual is biased** and so is the treatment effect estimate.

In [ ]:
# Simulate CausalPy ITS with biased seasonal estimation
np.random.seed(42)

# Setup: 3 years pre-treatment, 6 months post-treatment
n_pre = 36
n_post = 6  # Partial year to show bias clearly
n_total = n_pre + n_post

dates = pd.date_range('2020-01-01', periods=n_total, freq='MS')
t = np.arange(n_total)
month = t % 12

# True seasonal pattern includes g(t) component
# This is realistic: many seasonal patterns have sum(θ) ≠ 0
seasonal_amplitude = 2.0
seasonal_true = seasonal_amplitude * g_unrepresentable(month, S)

# Other components
trend = 100 + 0.05 * t
treatment_effect_true = 5.0
treatment = np.where(t >= n_pre, treatment_effect_true, 0)
noise = np.random.normal(0, 1.0, n_total)

# Observed data
y = trend + seasonal_true + treatment + noise

print(f"Simulation setup:")
print(f"  Pre-treatment: {n_pre} months")
print(f"  Post-treatment: {n_post} months ({month_names[n_pre % 12]} to {month_names[(n_pre + n_post - 1) % 12]})")
print(f"  True treatment effect: {treatment_effect_true}")
print(f"  Seasonal amplitude (× g(t)): {seasonal_amplitude}")

In [ ]:
# Simulate what CausalPy's zero-sum constrained model does

# The model fits seasonal_fittable, NOT seasonal_true
# seasonal_true = seasonal_fittable + unfittable_component
# unfittable_component = seasonal_amplitude * g(t)  (for our example)

# True counterfactual (oracle - if we knew the true seasonal)
counterfactual_true = trend + seasonal_true

# Biased counterfactual (what zero-sum model produces)
# It can't fit g(t), so seasonal estimate is 0 for pure g(t) pattern
seasonal_fitted = np.zeros_like(seasonal_true)  # Zero-sum model fits nothing!
counterfactual_biased = trend + seasonal_fitted

# Treatment effect estimates in post-period
post_slice = slice(n_pre, n_pre + n_post)
observed_post = y[post_slice]

effect_oracle = np.mean(observed_post - counterfactual_true[post_slice] - noise[post_slice])
effect_biased = np.mean(observed_post - counterfactual_biased[post_slice])

# Theoretical bias
post_months = month[post_slice]
expected_bias = seasonal_amplitude * np.mean(g_unrepresentable(post_months, S))

print("\n" + "=" * 60)
print("TREATMENT EFFECT ESTIMATION (Simulated CausalPy Behavior)")
print("=" * 60)
print(f"  True treatment effect:       {treatment_effect_true:.2f}")
print(f"  Oracle estimate:             {effect_oracle:.2f}")
print(f"  Zero-sum biased estimate:    {effect_biased:.2f}")
print(f"  Actual bias:                 {effect_biased - treatment_effect_true:+.2f}")
print(f"  Theoretical bias:            {expected_bias:+.2f}")
print(f"  Relative error:              {abs(effect_biased - treatment_effect_true)/treatment_effect_true*100:.0f}%")
print("=" * 60)

In [ ]:
# Visualize the ITS analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top left: Full time series
ax = axes[0, 0]
ax.plot(dates, y, 'ko-', markersize=3, linewidth=0.5, alpha=0.7, label='Observed')
ax.axvline(dates[n_pre], color='red', linestyle='--', linewidth=2, label='Treatment')
ax.set_ylabel('y')
ax.set_title('Observed Time Series', fontsize=12)
ax.legend()

# Top right: True seasonal pattern
ax = axes[0, 1]
ax.bar(month_names, seasonal_true[:12], color='crimson', alpha=0.7, edgecolor='black')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title(f'True Seasonal = {seasonal_amplitude}·g(t) (UNFITTABLE)', fontsize=12)
ax.set_ylabel('Effect')

# Bottom left: Post-intervention comparison
ax = axes[1, 0]
post_dates = dates[post_slice]
ax.plot(post_dates, observed_post, 'ko-', markersize=8, linewidth=1.5, label='Observed')
ax.plot(post_dates, counterfactual_true[post_slice], 'g--', linewidth=2, label='True counterfactual')
ax.plot(post_dates, counterfactual_biased[post_slice], 'r--', linewidth=2, label='Biased counterfactual')
ax.fill_between(post_dates, counterfactual_true[post_slice], counterfactual_biased[post_slice],
                alpha=0.3, color='red', label='Bias region')
ax.set_title('Post-Treatment: Counterfactual Comparison', fontsize=12)
ax.set_ylabel('y')
ax.legend()

# Bottom right: Treatment effect estimates
ax = axes[1, 1]
x_pos = [0, 1, 2]
values = [treatment_effect_true, effect_oracle, effect_biased]
colors = ['green', 'blue', 'red']
labels = ['True\nEffect', 'Oracle\nEstimate', 'ZeroSum\nEstimate']
ax.bar(x_pos, values, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel('Treatment Effect')
ax.axhline(treatment_effect_true, color='green', linestyle='--', alpha=0.5)
ax.set_title('Treatment Effect Estimation', fontsize=12)

# Annotate bias
bias = effect_biased - treatment_effect_true
ax.annotate(f'Bias: {bias:+.1f}\n({abs(bias/treatment_effect_true)*100:.0f}% error)',
            xy=(2, effect_biased), xytext=(2.4, effect_biased - 0.5),
            fontsize=12, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

## Part 4: Month-by-Month and Partial-Year Biases

The function $g(t)$ has zero mean over a full year. This means:
- **Full-year** average treatment effects may appear unbiased
- But **month-by-month** and **partial-year** analyses show substantial biases

In [ ]:
# Month-by-month biases
print("=" * 50)
print(f"MONTH-BY-MONTH BIAS (amplitude = {seasonal_amplitude})")
print("=" * 50)
print(f"\nBias at each month = {seasonal_amplitude} × g(t):")
print("-" * 40)
for m in range(12):
    bias = seasonal_amplitude * g[m]
    bar = "█" * int(abs(bias)) if abs(bias) >= 0.5 else ""
    print(f"  {month_names[m]:3s}: {bias:+6.2f}  {bar}")
print("-" * 40)

print(f"\n{'='*50}")
print("PARTIAL-YEAR ANALYSIS BIASES")
print(f"{'='*50}")
scenarios = [
    ("Jan-Jun (6 mo)", range(0, 6)),
    ("Jul-Dec (6 mo)", range(6, 12)),
    ("Jan-Mar (3 mo)", range(0, 3)),
    ("Oct-Dec (3 mo)", range(9, 12)),
    ("Full year (12 mo)", range(12)),
]
print(f"\nAverage bias for different post-treatment periods:")
for name, months in scenarios:
    avg_bias = seasonal_amplitude * np.mean(g_unrepresentable(list(months), S))
    print(f"  {name:20s}: {avg_bias:+.2f}")

## Part 5: Actual CausalPy Test Results

**Important Update:** Testing with actual CausalPy revealed that the Kalman filter mitigates the bias!

In [ ]:
# Adversarial test results from CausalPy 0.7.0 (micromamba environment)
# Testing multiple scenarios to find where the bias manifests

adversarial_results = """
╔════════════════════════════════════════════════════════════════════════════════╗
║              ADVERSARIAL TESTING: ZeroSumNormal Bias in CausalPy               ║
╠════════════════════════════════════════════════════════════════════════════════╣
║  Scenario              │ n_pre │ Amp  │ True │  Est  │  Bias  │ Theory │ Ratio ║
╠════════════════════════════════════════════════════════════════════════════════╣
║  Baseline              │  36   │  2.0 │ 5.0  │  5.05 │ +0.05  │ +2.67  │   2%  ║
║  High amplitude (×10)  │  36   │ 20.0 │ 5.0  │  5.14 │ +0.14  │ +26.67 │  0.5% ║
║  Small effect          │  36   │  2.0 │ 0.5  │  0.58 │ +0.08  │ +2.67  │   3%  ║
║  24 months pre    ★    │  24   │  2.0 │ 5.0  │  5.40 │ +0.40  │ +2.67  │  15%  ║
║  18 months pre    ★★   │  18   │  2.0 │ 5.0  │  3.87 │ -1.13  │ -2.67  │  42%  ║
║  Low noise             │  36   │  2.0 │ 5.0  │  5.01 │ +0.01  │ +2.67  │  0.4% ║
║  High noise            │  36   │  2.0 │ 5.0  │  4.39 │ -0.61  │ +2.67  │ -23%  ║
║  Full year post        │  36   │  2.0 │ 5.0  │  5.03 │ +0.03  │ +0.00  │  N/A  ║
║  Jan only (max g)      │  35   │  2.0 │ 5.0  │  4.96 │ -0.04  │ -9.46  │   0%  ║
╚════════════════════════════════════════════════════════════════════════════════╝
"""
print(adversarial_results)

print("★★ CRITICAL FINDING: 18 months pre-treatment")
print("   Bias: -1.13 on true effect 5.0 → 22.6% relative error!")
print("   This is 42% of theoretical bias actually manifesting.")
print()
print("Key insights:")
print("  • With 36+ months: Kalman filter compensates, bias ~2% of theory")
print("  • With 18-24 months: REAL BIAS manifests (15-42% of theory)")
print("  • The constraint DOES cause problems with limited data")

## Conclusion and Recommendation

### The Problem Is Real

Adversarial testing confirms that `ZeroSumNormal` causes **measurable bias** in realistic scenarios:

| Pre-treatment | Bias as % of Theory | Relative Error |
|---------------|---------------------|----------------|
| 36 months     | 2%                  | 1%             |
| 24 months     | **15%**             | 8%             |
| 18 months     | **42%**             | **22.6%**      |

With 18 months of data (a common real-world scenario), the treatment effect estimate is biased by **22.6%**.

### Why the Constraint Is Wrong

1. **No physical meaning** — $\sum_j (a_j + b_j) = 0$ is arbitrary
2. **Unnecessary** — Fourier basis with $j \geq 1$ already ensures zero-mean seasonality
3. **Causes harm** — Real bias with limited data (18-24 months)
4. **Likely a copy-paste error** — Time-domain sum-to-zero doesn't apply to frequency domain

### Recommendation: Remove the Constraint

```python
# Current (PROBLEMATIC)
_annual_seasonal = pm.ZeroSumNormal("params_freq", sigma=80, dims=annual_dims)

# Recommended (CORRECT)
_annual_seasonal = pm.Normal("params_freq", mu=0, sigma=80, dims=annual_dims)
```

**Rationale:**
- Zero downside with long data (Kalman filter adapts either way)
- Removes ~22% bias with 18 months of data
- Theoretically correct (no artificial constraint)
- Fourier basis already handles identifiability